In [21]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, io, warnings
from pathlib import Path
from scipy.signal import savgol_filter
sys.path.append(os.path.abspath('..'))
from src.wiserep import plot_spectra

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
target_name='SN2026jlm'
filedir=f'../output/{target_name}/spectrum/'
filepath=list(Path(filedir).glob('*.dat'))

plot_spectra(filepath, target_name, jupyter=True)

In [ ]:
import astrodash

# ── 抑制 astrodash / tensorflow / pandas 的冗长输出 ──
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示 TF 警告和错误
warnings.filterwarnings('ignore')           # 忽略 pandas/numpy 版本警告

# 抑制 astrodash 内部 print 的 20 行概率输出
import contextlib
class _SuppressPrint:
    def write(self, *a): pass
    def flush(self): pass
_orig_stdout = sys.stdout

filename_str = list(str(f) for f in filepath)

print(f'📂 光谱文件: {filename_str}')
print(f'✅ 文件存在: {Path(filename_str[0]).exists()}')

print("正在启动 DASH 进行模板匹配和深度学习分类...")

host_galaxy_z = 0.0022 #不知道就填 None

try:
    # 临时吞掉 astrodash 内部 print 的概率数组
    sys.stdout = io.StringIO()
    classification = astrodash.Classify(
        filenames=filename_str,
        knownZ=host_galaxy_z,
    )
    # 获取最佳匹配 (返回 6 元组)
    ret = classification.list_best_matches()
    sys.stdout = _orig_stdout  # 恢复标准输出
    
    # ret[0]=bestMatchLists, ret[1]=redshifts, ret[2]=bestBroadTypes,
    # ret[3]=rlapLabels, ret[4]=matchesReliableLabels, ret[5]=redshiftErrs
    best_fits = ret[2]          # 最佳匹配列表
    redshifts = ret[1]          # 拟合红移数组
    
    if best_fits is not None and len(best_fits) > 0:
        best_match = best_fits[0]
        sn_name = best_match[0]
        sn_type = best_match[1]
        sn_age  = best_match[2]
        
        # 从 redshifts 数组取第一个谱的最佳红移 (shape 可能是 (n,) 或 (1, n))
        if redshifts is not None and len(redshifts) > 0:
        # DASH 返回的 redshifts[0] 当 knownZ 指定时为 0，此时用 knownZ
            if host_galaxy_z is not None and redshifts is not None and len(redshifts) > 0:
                fitted_z = float(redshifts[0]) if redshifts.ndim == 1 else float(redshifts[0, 0])
                sn_z = fitted_z if abs(fitted_z) > 1e-6 else host_galaxy_z
            else:
                sn_z = None
        else:
            sn_z = None
        
        print("-" * 40)
        print(f"🎯 DASH 最终分类结果 (基于已知红移 z={host_galaxy_z}):")
        print(f"超新星类型 (Type): {sn_type}")
        print(f"演化阶段 (Age): {sn_age} days (相对于光变极大期)")
        if sn_z is not None:
            print(f"拟合红移 (z): {sn_z:.4f}")
        else:
            print(f"拟合红移 (z): 未提供")
        print(f"最匹配的标准模板: {sn_name}")
        print("-" * 40)

        # ── 拟合对比图：输入光谱 vs 最佳匹配模板 + 残差 ──
        try:
            _, _, _, inputImages, _ = classification._input_spectra_info()
            observed = inputImages[0]
            top_template = ret[0][0][0]  # (host, name, age, prob)
            tmpl_name = top_template[1]; tmpl_age = top_template[2]
            tmpl_prob = float(top_template[3])
            tmpl_flux = classification.snTemplates[tmpl_name][tmpl_age]["snInfo"][0][1]
            
            plt.figure(figsize=(10, 6))
            plt.plot(classification.wave, observed, "k-", lw=0.8, label="Observed (rebin)")
            npts = min(len(observed), len(tmpl_flux))
            plt.plot(classification.wave[:npts], tmpl_flux[:npts], "r-", lw=1.0, alpha=0.7,
                     label=f"Template: {tmpl_name} ({tmpl_age})")
            plt.ylabel("Normalized Flux")
            plt.title(f"DASH Fit: {sn_name} ({sn_type})  z={sn_z:.4f}")
            plt.legend(fontsize=9)
            plt.show()
        except Exception as e:
            print(f"(对比图生成失败: {e})")
    else:
        print("⚠️ DASH 未能找到可靠的匹配结果。")
except Exception as e:
    sys.stdout = _orig_stdout  # 确保恢复标准输出
    print(f"❌ DASH 分类失败: {e}")

In [ ]:
# ═════════════════════════════════════════════════════════
#  膨胀速度测量 — 默认使用 astrodash 的 DASH 分类结果
#  也可以手动输入 sn_z / rest_wave 覆盖自动值
# ═════════════════════════════════════════════════════════

# ── 1. 加载光谱数据 ──
# 从 cell 2 的 filepath 中取第一个 .dat 文件
spec_file = filepath[0]
data = np.loadtxt(spec_file, comments="#")
wave = data[:, 0]          # 观测波长 (Angstrom)
flux = data[:, 1]          # 流量
print(f"光谱文件: {spec_file.name}")
print(f"波长范围: {wave[0]:.1f} - {wave[-1]:.1f} A, 数据点: {len(wave)}")

# ── 2. 平滑光谱 ──
smooth_window = 15  # 平滑窗口 (必须是奇数)
flux_smoothed = savgol_filter(flux, smooth_window, polyorder=3)

# ── 3. 参数设定 ──
# 默认使用 astrodash 的结果；如需手动覆盖，取消注释并修改下面的值

# 红移 — 默认用 DASH 拟合值
# sn_z = 0.017              # <-- 手动指定红移，取消注释以覆盖 astrodash 结果
if "sn_z" not in dir() or sn_z is None:
    sn_z = 0.0              # astrodash 未运行或无红移时默认为 0
    print("⚠ sn_z 未定义，已设为 0 (请先运行 astrodash 或手动指定)")
else:
    print(f"✓ 使用 astrodash 的红移: sn_z = {sn_z}")

# 超新星类型 — 默认用 DASH 分类结果
# sn_type = "Ia"            # <-- 手动指定类型 (Ia / II / IIn / Ibc 等)
if "sn_type" not in dir() or sn_type is None:
    sn_type = "II"          # astrodash 未运行时默认为 II
    print("⚠ sn_type 未定义，默认设为 II")
else:
    print(f"✓ 使用 astrodash 的类型: sn_type = {sn_type}")

# ── 4. 根据类型选择特征谱线 ──
# DASH 返回的 sn_name 如 "Ia-norm" 包含类型；sn_type 是年龄范围
# 常见谱线静止波长 (Angstrom)，可手动修改 rest_wave
known_lines = {
    "Ha":   6562.8,  # 氢 Balmer-alpha  (II 型特征)
    "Hb":   4861.3,  # 氢 Balmer-beta
    "Si II": 6355.0,  # 硅吸收线        (Ia 型特征)
    "He I":  5875.6,  # 氦线            (Ibc 型特征)
    "Ca II": 3933.7,  # 钙 K 线
}

# 从 DASH 分类名推断类型 (sn_name 如 "Ia-norm", "II-norm", "Ic-broad")
dash_type = sn_name if sn_name else ""
if dash_type.lower().startswith("ia"):
    rest_wave = known_lines["Si II"]; line_name = "Si II"
elif dash_type.lower().startswith("ii") and not dash_type.lower().startswith("iin"):
    rest_wave = known_lines["Ha"];     line_name = "Ha"
elif dash_type.lower().startswith("iin"):
    rest_wave = known_lines["Ha"];     line_name = "Ha"
elif dash_type.lower().startswith("ib") or dash_type.lower().startswith("ic"):
    rest_wave = known_lines["He I"];   line_name = "He I"
else:
    rest_wave = known_lines["Ha"];     line_name = "Ha"  # 默认

# 手动覆盖: 取消注释下面这行

# ── 5. 计算膨胀速度 ──
c = 299792.458  # 光速 km/s

# 去红移，回到超新星静止参考系
wave_rest = wave / (1 + sn_z)
flux_rest = flux_smoothed  # 在 rest-frame 下形状不变

# 在静止波长附近的窗口搜索吸收谷
search_half = 400         # 搜索窗口 ±400 A
search_min = rest_wave - search_half
search_max = rest_wave + search_half

mask = (wave_rest > search_min) & (wave_rest < search_max)
local_wave = wave_rest[mask]
local_flux = flux_rest[mask]

if len(local_wave) == 0:
    print(f"❌ 搜索窗口内无数据点 (rest_wave={rest_wave})")
else:
    # 寻吸收谷: 在 rest_wave 蓝侧找最小流量
    blue_mask = local_wave < rest_wave
    if blue_mask.sum() > 5:
        blue_flux = local_flux[blue_mask]
        blue_wave = local_wave[blue_mask]
        min_idx = np.argmin(blue_flux)
        obs_abs_wave = blue_wave[min_idx]
    else:
        min_idx = np.argmin(local_flux)
        obs_abs_wave = local_wave[min_idx]

    v_exp = c * (rest_wave - obs_abs_wave) / rest_wave

    print()
    print("-" * 50)
    print(f"  谱线:      {line_name} (rest λ₀ = {rest_wave} A)")
    print(f"  测量蓝移:  λ_obs = {obs_abs_wave:.1f} A")
    print(f"  红移:      z = {sn_z:.4f}")
    print(f"  🚀 膨胀速度: v_exp = {v_exp:.0f} km/s")
    print("-" * 50)

    # ── 6. 绘图 ──
    plt.figure(figsize=(10, 6))
    plt.plot(wave_rest, flux_rest, "b-", lw=0.8, label="Smoothed")
    plt.axvspan(search_min, search_max, color="yellow", alpha=0.2, label="Search region")
    plt.axvline(rest_wave, color="red", ls="--", lw=1, label=f"{line_name} rest")
    plt.axvline(obs_abs_wave, color="green", ls=":", lw=1.5, label=f"Abs. minimum")
    plt.xlabel("Rest-frame Wavelength (A)")
    plt.ylabel("Flux")
    plt.title(f"{target_name} — {line_name} λ{rest_wave:.0f}  v={v_exp:.0f} km/s")
    plt.legend(fontsize=8)
    plt.show()